# 🔤 Byte Pair Encoding (BPE) Tokenizer — From Scratch

## What is BPE?

**Byte Pair Encoding (BPE)** is a subword tokenization algorithm originally invented for data compression, but repurposed for NLP by Sennrich et al. (2016). It is the backbone tokenizer behind:

- **GPT-2, GPT-3, GPT-4** (OpenAI)
- **LLaMA, Mistral** (Meta/Mistral AI)
- **RoBERTa, BART** (Facebook AI)

---

## 🧠 Core Intuition

The problem BPE solves:
- **Word-level tokenization** → huge vocabulary, can't handle unseen/rare words (`playing`, `played`, `plays` = 3 tokens)
- **Character-level tokenization** → tiny vocab, but sequences get very long
- **BPE** → finds the sweet spot: **subword units** that are frequent get merged into single tokens

**Key Idea:** Iteratively find the most frequent adjacent pair of symbols and merge them into a single new symbol.

---

## 📐 Algorithm Steps

```
1. Start: represent each word as a sequence of characters + special end-of-word token
2. Count: count frequency of all adjacent symbol pairs across the corpus
3. Merge: merge the most frequent pair into a new symbol
4. Update: update all word representations
5. Repeat: go to step 2 until you've done N merges (your hyperparameter)
6. Encode: to tokenize new text, apply the learned merges in order
```

---

## Visual Walkthrough

Corpus: `["low", "lower", "newest", "widest"]`

**Initial representation** (each word as chars, `</w>` marks word boundary):
```
l o w </w>       freq=5
l o w e r </w>   freq=2
n e w e s t </w> freq=6
w i d e s t </w> freq=3
```

**Step 1:** Most frequent pair = `(e, s)` → merge → `es`  
**Step 2:** Most frequent pair = `(es, t)` → merge → `est`  
**Step 3:** Most frequent pair = `(l, o)` → merge → `lo`  
...and so on


---
# PART 1: Tiny Corpus — Step-by-Step BPE by Hand

We'll implement every piece from scratch with detailed print statements so you can *see* each merge happening.

In [ ]:
# ── Imports (pure stdlib, no external deps needed for core BPE) ──
import re
from collections import defaultdict, Counter
import pprint


## Step 1 — Build Initial Vocabulary

Each word is split into characters. We append `</w>` (end-of-word marker) to distinguish, e.g., `low` appearing standalone vs. as prefix in `lower`.

In [ ]:
def get_vocab(corpus: list[str]) -> dict:
    """
    Build initial vocab from a corpus.
    Each word is turned into a space-separated sequence of chars + </w>.
    The dict maps this char-sequence → word frequency.
    
    Example:
        'low' (freq 5) → 'l o w </w>': 5
    """
    vocab = defaultdict(int)
    for word in corpus:
        # Split word into chars, add </w> at end, join with spaces
        token = ' '.join(list(word)) + ' </w>'
        vocab[token] += 1
    return dict(vocab)


# ── Our small corpus ──────────────────────────────────────────────
corpus = [
    'low', 'low', 'low', 'low', 'low',          # 'low'    appears 5x
    'lower', 'lower',                            # 'lower'  appears 2x
    'newest', 'newest', 'newest',                # 'newest' appears 3x  (changed for variety)
    'widest', 'widest',                          # 'widest' appears 2x
    'new', 'new', 'new', 'new',                  # 'new'    appears 4x
    'wide', 'wide',                              # 'wide'   appears 2x
]

vocab = get_vocab(corpus)

print("Initial vocabulary (char-sequence → frequency):")
print("=" * 55)
for token_seq, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f"  {freq:3d}×   {token_seq}")


## Step 2 — Count Pair Frequencies

For each word in our vocabulary, we look at every adjacent pair of symbols and count how often that pair appears (weighted by the word's frequency).

In [ ]:
def get_pair_stats(vocab: dict) -> dict:
    """
    Count frequency of every adjacent symbol pair across all words in vocab.
    
    vocab: {token_seq_string: frequency}
    returns: {(sym_a, sym_b): total_count}
    """
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()           # e.g. ['l','o','w','</w>']
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i+1])
            pairs[pair] += freq          # weighted by word frequency
    return dict(pairs)


pair_stats = get_pair_stats(vocab)

print("Top 10 most frequent pairs:")
print("=" * 40)
for pair, count in sorted(pair_stats.items(), key=lambda x: -x[1])[:10]:
    print(f"  {count:3d}×   {pair[0]} + {pair[1]}")


## Step 3 — Merge the Best Pair

We take the most frequent pair and replace every occurrence in the vocabulary with the merged symbol.

In [ ]:
def merge_vocab(pair: tuple, vocab: dict) -> dict:
    """
    Merge all occurrences of `pair` in vocab into a single new symbol.
    
    pair:  ('e', 's')  →  the two symbols to merge
    vocab: current vocabulary
    returns: new vocabulary with the pair merged everywhere
    
    Implementation trick:
      We use a regex that matches the pair separated by a space,
      but NOT preceded/followed by non-space (to avoid partial matches).
    """
    new_vocab = {}
    # Build regex: match ` e s ` but only as a whole symbol pair
    # re.escape handles special chars like '</w>'
    bigram = re.escape(' '.join(pair))          # e.g. 'e s'
    pattern = re.compile(r'(?<![\S])' + bigram + r'(?![\S])')
    merged_symbol = ''.join(pair)               # e.g. 'es'
    
    for word, freq in vocab.items():
        new_word = pattern.sub(merged_symbol, word)
        new_vocab[new_word] = freq
    return new_vocab


# ── Demo: do one manual merge ──────────────────────────────────────
best_pair = max(pair_stats, key=pair_stats.get)
print(f"Best pair to merge: {best_pair}  (freq={pair_stats[best_pair]})")

new_vocab = merge_vocab(best_pair, vocab)
print("\nVocab after first merge:")
for token_seq, freq in sorted(new_vocab.items(), key=lambda x: -x[1]):
    print(f"  {freq:3d}×   {token_seq}")


---
# PART 2: Full BPE Training Loop

Now we put it all together — train BPE for `N` merges and record every learned merge.

In [ ]:
def train_bpe(corpus: list[str], num_merges: int, verbose: bool = True):
    """
    Train BPE on a corpus for `num_merges` rounds.
    
    Returns:
        merges  : list of (pair_tuple, merged_symbol) in order applied
        vocab   : final vocabulary
        all_symbols: set of all symbols ever created (= your token inventory)
    """
    vocab = get_vocab(corpus)
    merges = []                  # ordered list of merges → the "merge table"
    all_symbols = set()          # grows with each merge
    
    # Seed all_symbols with initial chars
    for word in vocab:
        for sym in word.split():
            all_symbols.add(sym)
    
    if verbose:
        print(f"Starting vocab size (unique symbols): {len(all_symbols)}")
        print(f"Running {num_merges} BPE merges...")
        print("=" * 60)
    
    for i in range(num_merges):
        pair_stats = get_pair_stats(vocab)
        if not pair_stats:
            print("No more pairs to merge. Stopping early.")
            break
        
        # ── Pick the best pair ──────────────────────────────────────
        best_pair = max(pair_stats, key=pair_stats.get)
        best_count = pair_stats[best_pair]
        
        # ── Merge ───────────────────────────────────────────────────
        merged_symbol = ''.join(best_pair)
        vocab = merge_vocab(best_pair, vocab)
        merges.append((best_pair, merged_symbol))
        all_symbols.add(merged_symbol)
        
        if verbose:
            print(f"  Merge #{i+1:2d}: {best_pair[0]!r} + {best_pair[1]!r}"
                  f"  →  {merged_symbol!r}   (pair freq={best_count})")
    
    if verbose:
        print("=" * 60)
        print(f"Final vocab size (unique symbols): {len(all_symbols)}")
    
    return merges, vocab, all_symbols


# ── Train! ─────────────────────────────────────────────────────────
NUM_MERGES = 15
merges, final_vocab, all_symbols = train_bpe(corpus, num_merges=NUM_MERGES)


In [ ]:
# ── Inspect the final vocabulary ────────────────────────────────────
print("Final vocabulary (after all merges):")
print("=" * 55)
for token_seq, freq in sorted(final_vocab.items(), key=lambda x: -x[1]):
    tokens = token_seq.split()
    print(f"  {freq:3d}×   {token_seq:30s}  →  {tokens}")

print("\nAll symbols in inventory:")
print(sorted(all_symbols))


---
# PART 3: BPE Encoding (Tokenizing New Text)

Given the learned merge table, we can now tokenize **any new word** — even ones not in the training corpus.

**Algorithm:**
1. Split word into characters + `</w>`
2. Go through the merge table **in order** (order matters!)
3. Apply each merge if the pair exists in our current representation
4. Result = subword tokens

In [ ]:
def encode_word(word: str, merges: list, verbose: bool = False) -> list[str]:
    """
    Tokenize a single word using the learned BPE merge table.
    
    word:   raw word string (e.g. 'lowest')
    merges: ordered list of ((sym_a, sym_b), merged) from training
    returns: list of subword tokens
    """
    # Step 1: Initialize as char sequence
    symbols = list(word) + ['</w>']
    
    if verbose:
        print(f"\nEncoding: {word!r}")
        print(f"  Initial:  {symbols}")
    
    # Step 2: Apply merges in the learned order
    for (pair, merged) in merges:
        i = 0
        new_symbols = []
        while i < len(symbols):
            # Check if current + next match the pair
            if i < len(symbols) - 1 and (symbols[i], symbols[i+1]) == pair:
                new_symbols.append(merged)
                i += 2           # skip both symbols in the pair
            else:
                new_symbols.append(symbols[i])
                i += 1
        
        if new_symbols != symbols and verbose:
            print(f"  After merging {pair}: {new_symbols}")
        
        symbols = new_symbols
    
    return symbols


def encode(text: str, merges: list, verbose: bool = False) -> list[str]:
    """
    Tokenize a full string (splits on spaces first).
    """
    words = text.lower().split()
    all_tokens = []
    for word in words:
        tokens = encode_word(word, merges, verbose=verbose)
        all_tokens.extend(tokens)
    return all_tokens


# ── Test on known words ─────────────────────────────────────────────
for word in ['low', 'lower', 'newest', 'new', 'widest']:
    tokens = encode_word(word, merges, verbose=True)
    print(f"  RESULT → {tokens}")
    print()


In [ ]:
# ── Test on UNSEEN words — this is where BPE shines ────────────────
print("Testing BPE on words NOT in the training corpus:")
print("=" * 55)
unseen = ['lowest', 'newest', 'newest', 'newness', 'wild', 'wilder']
for word in unseen:
    tokens = encode_word(word, merges)
    print(f"  {word:15s} → {tokens}")


---
# PART 4: Visualizing the Merge Tree

Let's trace the complete history of merges as a readable table.

In [ ]:
print(f"{'#':>3}  {'Left':>8}  {'Right':>8}  →  {'Merged':>12}")
print("-" * 45)
for i, (pair, merged) in enumerate(merges, 1):
    print(f"  {i:2d}  {pair[0]:>8}  {pair[1]:>8}  →  {merged:>12}")


---
# PART 5: Decoding — Going Back to Text

BPE tokens are easy to decode: just concatenate them and strip `</w>`.

In [ ]:
def decode(tokens: list[str]) -> str:
    """
    Convert BPE tokens back to original string.
    </w> marks word boundary (replace with space or strip).
    """
    text = ''.join(tokens)
    # </w> = end of word → replace with space
    text = text.replace('</w>', ' ').strip()
    return text


# ── Round-trip test ─────────────────────────────────────────────────
test_phrases = [
    'low new wide',
    'lower newest widest',
    'new lower wide',
]
print("Round-trip encode → decode:")
print("=" * 50)
for phrase in test_phrases:
    tokens = encode(phrase, merges)
    reconstructed = decode(tokens)
    match = "✓" if reconstructed == phrase else "✗"
    print(f"  {match} Input   : {phrase!r}")
    print(f"    Tokens  : {tokens}")
    print(f"    Decoded : {reconstructed!r}")
    print()


---
# PART 6: Build a Full BPE Tokenizer Class

Let's wrap everything up into a clean, reusable class.

In [ ]:
class BPETokenizer:
    """
    A minimal but complete BPE tokenizer.
    
    Usage:
        tok = BPETokenizer(num_merges=20)
        tok.fit(corpus)
        tokens = tok.encode('hello world')
        text   = tok.decode(tokens)
    """
    
    def __init__(self, num_merges: int = 20):
        self.num_merges = num_merges
        self.merges = []          # ordered list of learned merges
        self.vocab = {}           # final char-sequence → freq vocab
        self.token2id = {}        # symbol → integer id
        self.id2token = {}        # integer id → symbol
        self._fitted = False
    
    # ── Private helpers (same as above) ────────────────────────────
    
    @staticmethod
    def _get_vocab(corpus):
        vocab = defaultdict(int)
        for word in corpus:
            token = ' '.join(list(word)) + ' </w>'
            vocab[token] += 1
        return dict(vocab)
    
    @staticmethod
    def _get_pair_stats(vocab):
        pairs = defaultdict(int)
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return dict(pairs)
    
    @staticmethod
    def _merge_vocab(pair, vocab):
        new_vocab = {}
        bigram = re.escape(' '.join(pair))
        pattern = re.compile(r'(?<![\S])' + bigram + r'(?![\S])')
        merged = ''.join(pair)
        for word, freq in vocab.items():
            new_vocab[pattern.sub(merged, word)] = freq
        return new_vocab
    
    # ── Public API ─────────────────────────────────────────────────
    
    def fit(self, corpus: list[str]):
        """Train BPE on a list of words."""
        self.vocab = self._get_vocab(corpus)
        self.merges = []
        all_symbols = set()
        
        # Seed with characters
        for word in self.vocab:
            for sym in word.split():
                all_symbols.add(sym)
        
        for _ in range(self.num_merges):
            stats = self._get_pair_stats(self.vocab)
            if not stats:
                break
            best = max(stats, key=stats.get)
            merged = ''.join(best)
            self.vocab = self._merge_vocab(best, self.vocab)
            self.merges.append((best, merged))
            all_symbols.add(merged)
        
        # Build token↔id mappings
        all_symbols.add('<unk>')   # unknown token for OOV chars
        sorted_syms = sorted(all_symbols)
        self.token2id = {sym: i for i, sym in enumerate(sorted_syms)}
        self.id2token = {i: sym for sym, i in self.token2id.items()}
        self._fitted = True
        print(f"BPETokenizer fitted. Vocabulary size: {len(self.token2id)} symbols.")
        return self
    
    def _encode_word(self, word: str) -> list[str]:
        symbols = list(word) + ['</w>']
        for (pair, merged) in self.merges:
            i, new_syms = 0, []
            while i < len(symbols):
                if i < len(symbols)-1 and (symbols[i], symbols[i+1]) == pair:
                    new_syms.append(merged)
                    i += 2
                else:
                    new_syms.append(symbols[i])
                    i += 1
            symbols = new_syms
        return symbols
    
    def encode(self, text: str, return_ids: bool = False):
        """Tokenize text. Returns tokens or integer ids."""
        assert self._fitted, "Call .fit() first!"
        tokens = []
        for word in text.lower().split():
            tokens.extend(self._encode_word(word))
        if return_ids:
            unk_id = self.token2id['<unk>']
            return [self.token2id.get(t, unk_id) for t in tokens]
        return tokens
    
    def decode(self, tokens_or_ids) -> str:
        """Convert tokens or ids back to string."""
        if tokens_or_ids and isinstance(tokens_or_ids[0], int):
            tokens = [self.id2token.get(i, '<unk>') for i in tokens_or_ids]
        else:
            tokens = tokens_or_ids
        return ''.join(tokens).replace('</w>', ' ').strip()
    
    def vocab_size(self) -> int:
        return len(self.token2id)
    
    def __repr__(self):
        status = f'fitted, vocab={self.vocab_size()}' if self._fitted else 'not fitted'
        return f'BPETokenizer(num_merges={self.num_merges}, {status})'


# ── Demonstrate the class ───────────────────────────────────────────
tok = BPETokenizer(num_merges=15)
tok.fit(corpus)

print()
for phrase in ['low new wide', 'lower newest widest', 'newest lower']:
    tokens = tok.encode(phrase)
    ids    = tok.encode(phrase, return_ids=True)
    back   = tok.decode(ids)
    print(f"Input   : {phrase!r}")
    print(f"Tokens  : {tokens}")
    print(f"IDs     : {ids}")
    print(f"Decoded : {back!r}")
    print()


---
# PART 7: Experiment — Effect of Number of Merges

More merges = larger vocabulary = longer tokens (less splitting).
Fewer merges = smaller vocabulary = more character-level splitting.

In [ ]:
test_word = 'widest'
print(f"How does the number of merges affect tokenization of {test_word!r}?")
print("=" * 55)

for n in [0, 2, 5, 10, 15, 20]:
    t = BPETokenizer(num_merges=n)
    t.fit(corpus)
    tokens = t.encode(test_word)
    print(f"  {n:3d} merges → {len(tokens):2d} tokens  {tokens}")


---
# PART 8: Real-World Scale Demo

Let's run BPE on a slightly larger corpus to see more interesting merges.

In [ ]:
# A small but realistic English vocabulary sample
larger_corpus = (
    ['the'] * 50 +
    ['and'] * 40 +
    ['that'] * 30 +
    ['have'] * 25 +
    ['for'] * 22 +
    ['not'] * 20 +
    ['with'] * 18 +
    ['you'] * 16 +
    ['this'] * 14 +
    ['but'] * 12 +
    ['running'] * 10 +
    ['runner'] * 8 +
    ['runs'] * 7 +
    ['playing'] * 9 +
    ['played'] * 8 +
    ['player'] * 7 +
    ['talk'] * 6 +
    ['talking'] * 5 +
    ['talked'] * 4 +
    ['learning'] * 10 +
    ['learned'] * 8 +
    ['learner'] * 6 +
    ['tokenize'] * 5 +
    ['tokenizer'] * 5 +
    ['tokenization'] * 4
)

big_tok = BPETokenizer(num_merges=30)
big_tok.fit(larger_corpus)

print()
test_words = ['running', 'runner', 'runs', 'played', 'player', 'tokenizer',
              'tokenization', 'unlearned', 'relearning']
print(f"{'Word':20s}  {'Tokens':50s}  #tok")
print("-" * 80)
for w in test_words:
    t = big_tok.encode(w)
    print(f"  {w:18s}  {str(t):50s}  {len(t)}")


---
# PART 9: Key Concepts Summary

## Why `</w>` (end-of-word marker)?

Consider the characters `e` and `r` appearing in:
- `lower` → `er` at word **end**
- `era` → `er` at word **start**

Without the marker, `er` would merge regardless of position. With `</w>`, the end-of-`lower` becomes `er</w>`, which is a **different pair** from `er` in `era`. This gives BPE positional awareness.

## Why Order of Merges Matters

When encoding, merges are applied **in the exact order they were learned**. If merge #3 was `(e, s) → es` and merge #7 was `(es, t) → est`, then:
- `e s t` → after merge #3: `es t` → after merge #7: `est` ✓
- If you tried merge #7 first, `(e, s, t)` has no `(es, t)` pair yet → wrong result ✗

## BPE vs Other Tokenizers

| Method | Vocab Size | OOV handling | Used by |
|--------|-----------|--------------|--------|
| Word-level | 50k-500k | ❌ `<unk>` | old NLP |
| Char-level | ~256 | ✅ always | some older models |
| **BPE** | 8k-50k | ✅ falls back to chars | GPT-2/3/4, LLaMA |
| WordPiece | 8k-30k | ✅ | BERT |
| SentencePiece | 8k-64k | ✅ | T5, XLNet |

## The Magic Number: 50,257

GPT-2 uses exactly **50,257 tokens**: 50,000 BPE merges + 256 base bytes + 1 special `<|endoftext|>` token.

In [ ]:
# ── Final sanity check: everything in one cell ──────────────────────
print("🎯 BPE Tokenizer — Quick Demo")
print("=" * 50)

mini_corpus = ['hello'] * 5 + ['hell'] * 3 + ['world'] * 4 + ['word'] * 2 + ['work'] * 3
t = BPETokenizer(num_merges=10)
t.fit(mini_corpus)

for w in ['hello', 'world', 'hellish', 'worldwide', 'helloworld']:
    print(f"  {w:15s} → {t.encode(w)}")

print("\nMerge table:")
for i, (pair, merged) in enumerate(t.merges, 1):
    print(f"  {i:2}. {pair[0]} + {pair[1]} → {merged}")


---
## 📚 Further Reading

- **Original BPE paper**: Sennrich et al. (2016) — *Neural Machine Translation of Rare Words with Subword Units*
- **GPT-2 tokenizer**: `tiktoken` library by OpenAI
- **HuggingFace tokenizers**: `tokenizers` library (Rust implementation, very fast)
- **Andrej Karpathy's minBPE**: A clean from-scratch Python implementation

## 🔑 Key Takeaways

1. **BPE is greedy** — always merges the globally most frequent pair
2. **Training** = learning a merge table; **Encoding** = applying that table in order
3. **`</w>` marker** distinguishes word-boundary characters from interior ones
4. **More merges** → larger tokens, more compressed representation
5. **Never produces `<unk>`** (for known scripts) — always falls back to individual chars